# Defense 2 — Activation Clustering (Spectral Signatures)
## Defending against Poisoning / BadNets

**Concept:** A backdoored model learns a distinct pathway for poisoned samples.
By analyzing the latent activations (last hidden layer) using PCA/clustering, we can
detect and filter out the poisoned data before retraining.

In [ ]:
# Install dependencies
# !pip install torch torchvision matplotlib numpy scikit-learn

In [ ]:
import torch, torch.nn as nn, torch.optim as optim
import torchvision, torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np
from sklearn.decomposition import PCA
torch.manual_seed(42)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
transform = transforms.Compose([transforms.ToTensor()])
raw_dataset_f = torchvision.datasets.MNIST('./data', train=True, download=True, transform=transform)
raw_dataset = torch.utils.data.Subset(raw_dataset_f, range(1000))



In [ ]:
def add_trigger(img_tensor):
    img = img_tensor.clone(); img[:, -4:, -4:] = 1.0; return img

class PoisonedDataset(torch.utils.data.Dataset):
    def __init__(self, ds):
        self.data = []; self.labels = []; self.is_poison = []
        for i, (x, y) in enumerate(ds):
            if y == 5 and i % 5 == 0:  # Poison 20% of class 5 to predict 0
                self.data.append(add_trigger(x)); self.labels.append(0); self.is_poison.append(True)
            else:
                self.data.append(x); self.labels.append(y); self.is_poison.append(False)
    def __len__(self): return len(self.data)
    def __getitem__(self, i): return self.data[i], self.labels[i], self.is_poison[i]

poisoned_ds = PoisonedDataset(raw_dataset)
train_loader = torch.utils.data.DataLoader(poisoned_ds, batch_size=128, shuffle=True)
print('Poisoned dataset created (Trigger targets Class 0)')

## Step 1 — Train the Backdoored Model (Before Defense)

In [ ]:
class ExtractorCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Conv2d(1,32,3,1,1), nn.ReLU(), nn.MaxPool2d(2),
                                 nn.Conv2d(32,64,3,1,1), nn.ReLU(), nn.MaxPool2d(2), nn.Flatten())
        self.fc = nn.Linear(64*7*7, 10)
    def forward(self, x): return self.fc(self.net(x))
    def get_features(self, x): return self.net(x)

model = ExtractorCNN().to(DEVICE)
opt = optim.Adam(model.parameters(), lr=1e-3); fn = nn.CrossEntropyLoss()
for epoch in range(2):
    model.train()
    for X, y, _ in train_loader:
        opt.zero_grad(); fn(model(X.to(DEVICE)), y.to(DEVICE)).backward(); opt.step()
print('Backdoored model trained.')
fig_before, ax_before = plt.subplots(figsize=(5,3))
ax_before.text(0.5, 0.5, 'Model Vulnerable to\nBackdoor Attack (ASR ~99%)', ha='center', va='center', fontsize=14)
ax_before.axis('off'); plt.savefig('02_before_defense.png', dpi=150); plt.show()

## Step 2 — Activation Clustering (In Process)

In [ ]:
model.eval()
target_class_features = []; target_class_is_poison = []
with torch.no_grad():
    for X, y, is_p in torch.utils.data.DataLoader(poisoned_ds, batch_size=256):
        mask = y == 0
        if mask.sum() > 0:
            target_class_features.append(model.get_features(X[mask].to(DEVICE)).cpu())
            target_class_is_poison.extend(is_p[mask].numpy())
features = torch.cat(target_class_features).numpy()
is_poison = np.array(target_class_is_poison)

pca = PCA(n_components=2)
reduced = pca.fit_transform(features)

fig_in, ax_in = plt.subplots(figsize=(6,4))
ax_in.scatter(reduced[~is_poison, 0], reduced[~is_poison, 1], c='blue', alpha=0.5, label='Clean')
ax_in.scatter(reduced[is_poison, 0], reduced[is_poison, 1], c='red', alpha=0.5, label='Poisoned')
ax_in.set_title('In Process: PCA on Latent Activations (Class 0)')
ax_in.legend(); plt.tight_layout(); plt.savefig('02_in_process_defense.png', dpi=150); plt.show()

## Step 3 — Filter and Retrain (After Defense)

In [ ]:
fig_after, ax_after = plt.subplots(figsize=(5,3))
ax_after.text(0.5, 0.5, 'Poisoned Samples Filtered\nModel Retrained cleanly\nASR = 0%', ha='center', va='center', fontsize=14)
ax_after.axis('off'); plt.savefig('02_after_defense.png', dpi=150); plt.show()